# Playground Series - EDA & 5 Models Comparison
## Predicting Customer Churn - Playground Series S6E3
### Kaggle Playground Series · Season 6 Episode 3 | March 2026

**Competition Goal:** Given customer information from a telecom company, predict the probability that a customer will churn (i.e., leave/cancel their subscription).

**Evaluation Metric:** ROC-AUC (Area Under the Receiver Operating Characteristic Curve). Higher is better. A score of 0.5 = random guessing; 1.0 = perfect predictions.

**Business Context:** Acquiring a new customer costs 5–25× more than retaining an existing one. If a company can predict who will leave, they can intervene early with promotions, better plans, or targeted support.

---

## 🎯 Key Hypotheses Before Modeling
Based on domain knowledge, we expect:
- **Short tenure + Month-to-month contract** → higher churn (no lock-in)
- **Fiber optic users** may churn more (higher cost, more competition)
- **No add-on services** (no security, no backup) → easier to leave
- **Higher monthly charges** with no discounts → price-sensitive churn
- **Electronic check payments** may correlate with less committed customers

## 📊 Models We Will Compare
| Model | Why Include It | Prediction |
|-------|---|---|
| Logistic Regression | Interpretable baseline; works well on linear relationships | Strong baseline |
| Decision Tree | Visualizable; captures non-linear splits | Good interpretability |
| Random Forest | Ensemble of trees; reduces overfitting | Very good |
| XGBoost | Gradient boosting; often best on tabular data | **Likely winner** |
| LightGBM | Faster gradient boosting; great with categorical features | **Likely best** |

# Playground Series - EDA & 5 Models Comparison
## Predicting Customer Churn - Playground Series S6E3
### Kaggle Playground Series · Season 6 Episode 3 | March 2026

**Competition Goal:** Given customer information from a telecom company, predict the probability that a customer will churn (i.e., leave/cancel their subscription).

**Evaluation Metric:** ROC-AUC (Area Under the Receiver Operating Characteristic Curve). Higher is better. A score of 0.5 = random guessing; 1.0 = perfect predictions.

**Business Context:** Acquiring a new customer costs 5–25× more than retaining an existing one. If a company can predict who will leave, they can intervene early with promotions, better plans, or targeted support.

## 1️⃣ File Check & Library Import

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import os, warnings, pickle
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Set a clean style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = {'No': '#4E79A7', 'Yes': '#E15759'}  # blue = no churn, red = churn

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

# Preprocessing & pipelines 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Model selection & evaluation
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

# SHAP for explainability
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('⚠️ Install shap for explainability plots: pip install shap')

print('✅ All libraries loaded successfully!')

## 2️⃣ Exploratory Data Analysis (EDA)

### Load Data & Overview

In [ ]:
# Load datasets
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nFirst few rows:\n')
print(train.head())

# Check for missing values
missing = train.isnull().sum()
print(f'\n\nMissing values: {missing.sum() if missing.sum() > 0 else "None — clean dataset! ✅"}')
print(f'Duplicate rows: {train.duplicated().sum()}')

### Target Variable Distribution & Class Balance

In [ ]:
churn_counts = train['Churn'].value_counts()
churn_pct    = train['Churn'].value_counts(normalize=True) * 100

print('Churn distribution:')
print(pd.DataFrame({'count': churn_counts, 'percent': churn_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
sns.countplot(data=train, x='Churn', palette=PALETTE, ax=axes[0])
axes[0].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[0].bar_label(axes[0].containers[0], fmt='%d')

# Pie chart
axes[1].pie(churn_counts, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#4E79A7', '#E15759'],
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n⚠️ NOTE: Class imbalance (~77% No Churn, ~23% Yes Churn).')
print('ROC-AUC handles imbalance well, but we should monitor this.')

### Numerical Feature Distributions

In [ ]:
# Fix TotalCharges (stored as string in raw data)
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce')
test['TotalCharges']  = pd.to_numeric(test['TotalCharges'],  errors='coerce')

# New customers (tenure=0) have TotalCharges=NaN → fill with 0
train['TotalCharges'].fillna(0, inplace=True)
test['TotalCharges'].fillna(0, inplace=True)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, col in enumerate(num_cols):
    # KDE by churn
    sns.kdeplot(data=train, x=col, hue='Churn', hue_order=['No','Yes'],
                fill=True, alpha=0.4, palette=PALETTE, ax=axes[0, i])
    axes[0, i].set_title(f'{col} — KDE by Churn', fontsize=12)
    
    # Box plot by churn
    sns.boxplot(data=train, y=col, x='Churn', order=['No','Yes'],
                palette=PALETTE, ax=axes[1, i])
    axes[1, i].set_title(f'{col} — Box Plot by Churn', fontsize=12)

plt.suptitle('Numerical Feature Distributions vs Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('🔍 Key Observations:')
print('• Short-tenure customers churn significantly more')
print('• Churned customers have higher monthly charges on average')
print('• Non-churners have higher total charges (they\'ve stayed longer)')

### Key Insights: Contract, Tenure, and Charges

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Churn rate by Contract type
contract_churn = (train.groupby('Contract')['Churn']
                  .apply(lambda x: (x=='Yes').mean()*100).reset_index())
sns.barplot(data=contract_churn, x='Contract', y='Churn',
            palette=['#E15759','#F28E2B','#4E79A7'], ax=axes[0])
axes[0].set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                    (p.get_x()+p.get_width()/2, p.get_height()+0.3),
                    ha='center', fontsize=10)

# 2. Tenure distribution by churn
bins = [0, 12, 24, 36, 48, 60, 72]
labels_t = ['0-12m','13-24m','25-36m','37-48m','49-60m','61-72m']
train['tenure_group'] = pd.cut(train['tenure'], bins=bins, labels=labels_t)
tg_churn = (train.groupby('tenure_group', observed=True)['Churn']
            .apply(lambda x: (x=='Yes').mean()*100).reset_index())
sns.barplot(data=tg_churn, x='tenure_group', y='Churn', palette='Blues_r', ax=axes[1])
axes[1].set_title('Churn Rate by Tenure Group', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('Tenure Group')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                    (p.get_x()+p.get_width()/2, p.get_height()+0.3),
                    ha='center', fontsize=9)

# 3. Monthly charges vs tenure scatter
axes[2].scatter(
    train[train['Churn']=='No']['tenure'],
    train[train['Churn']=='No']['MonthlyCharges'],
    alpha=0.2, c='#4E79A7', label='No Churn', s=5
)
axes[2].scatter(
    train[train['Churn']=='Yes']['tenure'],
    train[train['Churn']=='Yes']['MonthlyCharges'],
    alpha=0.4, c='#E15759', label='Churn', s=5
)
axes[2].set_title('Tenure vs Monthly Charges (by Churn)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Tenure (months)')
axes[2].set_ylabel('Monthly Charges ($)')
axes[2].legend(markerscale=3)

plt.tight_layout()
plt.show()

print('\n🔑 Key Findings:')
print('  1. Month-to-month customers churn ~3x more than 2-year contract customers')
print('  2. New customers (0-12 months) have the highest churn rate')
print('  3. High monthly charges + short tenure = highest churn risk zone')

### Correlation Analysis

In [ ]:
# Temporary encode for correlation
temp = train.copy()
temp['Churn_enc'] = (temp['Churn'] == 'Yes').astype(int)
for col in temp.select_dtypes(include='object').columns:
    temp[col] = LabelEncoder().fit_transform(temp[col].astype(str))
temp = temp.drop(columns=['id', 'tenure_group'], errors='ignore')

plt.figure(figsize=(14, 10))
corr = temp.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            annot_kws={'size':8})
plt.title('Correlation Heatmap (lower triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with Churn
churn_corr = corr['Churn_enc'].drop('Churn_enc').abs().sort_values(ascending=False)
print('\nTop 10 features correlated with Churn:')
print(churn_corr.head(10).to_string())
print('\n📌 Note: Label-encoded correlations on categorical variables can be misleading.')
print('Treat these as directional signals, not definitive rankings.')

---

## 3️⃣ Data Preprocessing & Feature Engineering

### Preprocessing Strategy:
- **Drop:** customerID
- **Binary Yes/No:** Map to 1/0
- **Multi-category:** One-hot encode
- **Numeric:** Keep as-is (scale only for Logistic Regression)
- **New Features:**
  - `AvgMonthlyCharge` = TotalCharges / tenure
  - `ChargeRatio` = MonthlyCharges / AvgMonthlyCharge
  - `IsNewCustomer` = 1 if tenure ≤ 12 months
  - `IsLongTerm` = 1 if tenure ≥ 60 months
  - `NumAddOns` = count of paid add-on services

In [ ]:
def preprocess(df, is_train=True):
    df = df.copy()
    
    # Drop ID
    df.drop(columns=['customerID'], inplace=True, errors='ignore')
    
    # Fix TotalCharges
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    
    # Target encode
    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    
    # Binary Yes/No columns 
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] == 'Yes').astype(int)
    
    # Gender
    df['gender'] = (df['gender'] == 'Male').astype(int)
    
    # Multi-category One-Hot Encoding
    ohe_cols = [
        'MultipleLines', 'InternetService',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies',
        'Contract', 'PaymentMethod'
    ]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=int)
    
    # Feature Engineering
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'].clip(lower=1))
    df['ChargeRatio']      = df['MonthlyCharges'] / (df['AvgMonthlyCharge'].clip(lower=0.01))
    df['IsNewCustomer']    = (df['tenure'] <= 12).astype(int)
    df['IsLongTerm']       = (df['tenure'] >= 60).astype(int)
    
    # Count add-on services
    addon_cols = [c for c in df.columns if any(s in c for s in
                  ['OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
                   'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes'])]
    df['NumAddOns'] = df[addon_cols].sum(axis=1) if addon_cols else 0
    
    return df

# Apply preprocessing
train_clean = preprocess(train.drop(columns=['tenure_group'], errors='ignore'), is_train=True)
test_clean  = preprocess(test, is_train=False)

# Align columns (test may be missing some dummy cols)
missing_cols = set(train_clean.columns) - set(test_clean.columns) - {'Churn'}
for col in missing_cols:
    test_clean[col] = 0
test_clean = test_clean[train_clean.drop(columns='Churn').columns]

print(f'✅ Train clean shape: {train_clean.shape}')
print(f'✅ Test clean shape:  {test_clean.shape}')
print(f'\n🆕 New features added: AvgMonthlyCharge, ChargeRatio, IsNewCustomer, IsLongTerm, NumAddOns')

In [ ]:
# Train/test split
y = train_clean['Churn']
X = train_clean.drop(columns=['Churn'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train churn rate: {y_train.mean():.2%}')
print(f'y_test  churn rate: {y_test.mean():.2%}')

---

## 4️⃣ Model Training & Evaluation

### Helper Functions

In [ ]:
RESULTS = []

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Fit a model and return a results row."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    row = {
        'Model': name,
        'ROC-AUC': round(roc_auc_score(y_te, y_prob), 5),
        'Accuracy': round(accuracy_score(y_te, y_pred), 5),
        'Precision': round(precision_score(y_te, y_pred), 5),
        'Recall': round(recall_score(y_te, y_pred), 5),
        'F1': round(f1_score(y_te, y_pred), 5),
    }
    RESULTS.append(row)
    print(f'✅ {name}: ROC-AUC = {row["ROC-AUC"]}')
    return model, y_prob

def show_confusion(model, X_te, y_te, name):
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_te, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn','Churn'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'{name} — Confusion Matrix')
    plt.tight_layout()
    plt.show()

print('✅ Helper functions ready')

### Model 1: Logistic Regression
Linear classifier with fast training and good interpretability. Requires feature scaling.

In [ ]:
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge', 'ChargeRatio']

X_train_lr = X_train.copy()
X_test_lr  = X_test.copy()

scaler = StandardScaler()
X_train_lr[scale_cols] = scaler.fit_transform(X_train_lr[scale_cols])
X_test_lr[scale_cols]  = scaler.transform(X_test_lr[scale_cols])

lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr_fit, lr_probs = evaluate_model('Logistic Regression', lr,
                                   X_train_lr, X_test_lr, y_train, y_test)
show_confusion(lr_fit, X_test_lr, y_test, 'Logistic Regression')

### Model 2: Decision Tree
Single interpretable tree with visualizable decision paths. Use GridSearchCV to tune hyperparameters.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)

cv_params = {
    'max_depth': [5, 8, 12],
    'min_samples_leaf': [20, 50, 100],
    'criterion': ['gini', 'entropy']
}

tree_cv = GridSearchCV(tree, cv_params,
                       scoring='roc_auc', cv=5,
                       refit=True, n_jobs=-1, verbose=0)
tree_cv.fit(X_train, y_train)

print(f'Best params: {tree_cv.best_params_}')
print(f'Best CV ROC-AUC: {tree_cv.best_score_:.5f}')

tree_fit, tree_probs = evaluate_model('Decision Tree', tree_cv.best_estimator_,
                                       X_train, X_test, y_train, y_test)
show_confusion(tree_fit, X_test, y_test, 'Decision Tree')

# Visualize top of decision tree
plt.figure(figsize=(20, 8))
plot_tree(tree_cv.best_estimator_, max_depth=3, fontsize=10,
          feature_names=X_train.columns,
          class_names=['No Churn', 'Churn'],
          filled=True, impurity=False, rounded=True)
plt.title('Decision Tree (top 3 levels)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Model 3: Random Forest
Ensemble of bootstrapped trees with random feature selection. More robust and lower variance than single tree.

In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

cv_params = {
    'n_estimators': [100],    
    'max_depth': [15],        
    'min_samples_leaf': [10], 
    'max_features': ['sqrt']  
}

rf_cv = GridSearchCV(rf, cv_params,
                     scoring='roc_auc', cv=4,
                     refit=True, n_jobs=-1, verbose=0)
rf_cv.fit(X_train, y_train)

print(f'Best params: {rf_cv.best_params_}')
print(f'Best CV ROC-AUC: {rf_cv.best_score_:.5f}')

rf_fit, rf_probs = evaluate_model('Random Forest', rf_cv.best_estimator_,
                                   X_train, X_test, y_train, y_test)
show_confusion(rf_fit, X_test, y_test, 'Random Forest')

### Model 4: XGBoost
Gradient boosting with L1/L2 regularization. Sequentially builds trees to correct previous errors.

In [ ]:
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    tree_method='hist'
)

cv_params = {
    'n_estimators':    [500, 1000],
    'max_depth':       [4, 6],
    'learning_rate':   [0.05, 0.1],
    'min_child_weight':[3, 5],
    'subsample':       [0.8],
    'colsample_bytree':[0.8]
}

xgb_cv = GridSearchCV(xgb, cv_params,
                      scoring='roc_auc', cv=4,
                      refit=True, n_jobs=-1, verbose=0)
xgb_cv.fit(X_train, y_train)

print(f'Best params: {xgb_cv.best_params_}')
print(f'Best CV ROC-AUC: {xgb_cv.best_score_:.5f}')

xgb_fit, xgb_probs = evaluate_model('XGBoost', xgb_cv.best_estimator_,
                                      X_train, X_test, y_train, y_test)
show_confusion(xgb_fit, X_test, y_test, 'XGBoost')

### Model 5: LightGBM (Optional)
Histogram-based gradient boosting with native categorical support. Fastest and often best AUC on tabular data.

In [ ]:
print('⏭️  Skipping LightGBM — Using XGBoost as Champion')
print('🏆 XGBoost ROC-AUC = 0.91689')

---

## 5️⃣ Model Comparison & Champion Selection

In [ ]:
results_df = pd.DataFrame(RESULTS).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results_df.index += 1

print('='*70)
print('🏆 MODEL COMPARISON (Holdout Test Set — 25% of Train Data)')
print('='*70)
print(results_df.to_string())
print('='*70)
print(f'\n🥇 CHAMPION: {results_df.iloc[0]["Model"]}')
print(f'   ROC-AUC: {results_df.iloc[0]["ROC-AUC"]}')

# Map champion to model object
champion_map = {
    'Logistic Regression': lr_fit,
    'Decision Tree':       tree_fit,
    'Random Forest':       rf_fit,
    'XGBoost':             xgb_fit,
}

champion_name  = results_df.iloc[0]['Model']
champion_model = champion_map[champion_name]
print(f'\n✅ Champion model object loaded: {champion_name}')

In [ ]:
# ROC Curve comparison
model_probs = {
    'Logistic Regression': (lr_fit, X_test_lr),
    'Decision Tree':       (tree_fit, X_test),
    'Random Forest':       (rf_fit, X_test),
    'XGBoost':             (xgb_fit, X_test),
}

colors = ['#4E79A7', '#F28E2B', '#59A14F', '#E15759']

plt.figure(figsize=(9, 7))
for (name, (model, X_te)), color in zip(model_probs.items(), colors):
    probs = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, lw=2.5, color=color, label=f'{name} (AUC={auc:.4f})')

plt.plot([0,1],[0,1],'k--', lw=1.5, label='Random (AUC=0.500)')
plt.fill_between([0,1],[0,1], alpha=0.05, color='grey')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — All Models Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Bar chart of ROC-AUC scores
plt.figure(figsize=(9, 5))
bars = plt.bar(results_df['Model'], results_df['ROC-AUC'],
               color=colors[::-1], edgecolor='white', linewidth=1.5)

# Gold border on champion
bars[0].set_edgecolor('gold')
bars[0].set_linewidth(3)

# Add value labels
for bar, val in zip(bars, results_df['ROC-AUC']):
    plt.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + 0.001,
             f'{val:.4f}', 
             ha='center', va='bottom', 
             fontweight='bold', fontsize=10)

plt.ylim(0.88, results_df['ROC-AUC'].max() + 0.02)
plt.ylabel('ROC-AUC Score', fontsize=12)
plt.title('Model ROC-AUC Comparison (🏆 = Champion)', fontsize=14, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---

## 6️⃣ Feature Importance & SHAP Analysis

SHAP (SHapley Additive exPlanations) explains why the model makes each prediction, not just which features are important overall.

In [ ]:
### Standard Feature Importance

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

models_fi = [
    ('Decision Tree', tree_fit, X_train),
    ('Random Forest', rf_fit, X_train),
    ('XGBoost', xgb_fit, X_train)
]

all_importances = {}
for ax, (name, model, X_tr) in zip(axes, models_fi):
    fi = pd.Series(model.feature_importances_, index=X_tr.columns)
    fi = fi.sort_values(ascending=False).head(15)
    all_importances[name] = fi
    
    sns.barplot(x=fi.values, y=fi.index, palette='Blues_r', ax=ax)
    ax.set_title(f'{name}\nTop 15 Features', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# XGBoost feature importance (champion)
fi_xgb = pd.Series(
    xgb_fit.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
colors_fi = ['#E15759' if i < 3 else '#4E79A7' for i in range(len(fi_xgb))]
sns.barplot(x=fi_xgb.values, y=fi_xgb.index, palette=colors_fi)
plt.title('XGBoost 🏆 — Top 20 Feature Importances (🔴 = Top 3)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\n✨ Top 10 XGBoost Features:')
print(fi_xgb.head(10).to_string())

In [ ]:
### SHAP Analysis (Optional)

if SHAP_AVAILABLE:
    X_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
    
    explainer   = shap.TreeExplainer(xgb_fit)  # ← XGBoost champion
    shap_values = explainer.shap_values(X_sample)
    
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    # Summary bar plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_sample, plot_type='bar', 
                      show=False, max_display=15, 
                      color='#4E79A7')
    plt.title('SHAP Feature Importance — XGBoost 🏆', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Beeswarm plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_sample, show=False, max_display=15)
    plt.title('SHAP Beeswarm — XGBoost 🏆', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ SHAP not available. Install with: pip install shap')

---

## 7️⃣ Final Prediction & Submission

### Retrain Champion on Full Training Data

In [ ]:
# Identify champion and prepare data
champion_name = results_df.iloc[0]['Model']
print(f'🏆 Champion: {champion_name}')

champion_map_final = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, C=1.0),
    'Decision Tree':       tree_cv.best_estimator_,
    'Random Forest':       rf_cv.best_estimator_,
    'XGBoost':             xgb_cv.best_estimator_,
}

if champion_name not in champion_map_final:
    print(f'⚠️ {champion_name} not available — falling back to XGBoost')
    champion_name = 'XGBoost'

champion = champion_map_final[champion_name]
print(f'✅ Using: {champion_name}')

# For LR, scale the full data
if champion_name == 'Logistic Regression':
    X_full      = X.copy()
    test_final  = test_clean.copy()
    X_full[scale_cols]     = scaler.fit_transform(X_full[scale_cols])
    test_final[scale_cols] = scaler.transform(test_final[scale_cols])
else:
    X_full     = X.copy()
    test_final = test_clean.copy()

# Refit on full training data
champion.fit(X_full, y)
print(f'✅ Champion refit on full training data ({len(X_full):,} rows)')

In [ ]:
# Generate predictions
test_probs = champion.predict_proba(test_final)[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    'Churn': test_probs
})

submission.to_csv('submission.csv', index=False)

print('📤 submission.csv saved!')
print(f'Rows: {len(submission)}')
print(f'Predicted churn probability — mean: {test_probs.mean():.3f}, std: {test_probs.std():.3f}')
print('\nFirst 5 predictions:')
print(submission.head())

In [ ]:
# Distribution of predicted probabilities
plt.figure(figsize=(9, 4))
plt.hist(test_probs, bins=50, color='#4E79A7', edgecolor='white', alpha=0.85)
plt.axvline(0.5, color='#E15759', lw=2, linestyle='--', label='0.5 threshold')
plt.xlabel('Predicted Churn Probability', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Predicted Churn Probabilities (Test Set)', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'\n📊 Customers predicted to churn (prob > 0.5): {(test_probs > 0.5).sum():,} ({(test_probs > 0.5).mean():.1%})')

---

## 8️⃣ Conclusion & Business Insights

### 📊 Model Performance Summary

| Model | Key Strength | ROC-AUC | Notes |
|-------|---|---|---|
| Logistic Regression | Interpretable, fast, strong baseline | 0.90895 | Requires scaling |
| Decision Tree | Fully visualizable decision paths | 0.91002 | Good explainability |
| Random Forest | Robust, low variance ensemble | 0.91453 | Slower to train |
| **XGBoost** 🏆 | **Excellent on tabular data** | **0.91689** | **Champion model** |
| LightGBM | Fastest, native categorical support | — | Could match or beat XGBoost |

### 🔑 Key Churn Drivers (All Models Agree)

All models consistently rank these as most important predictors:

1. **Contract Type** — Month-to-month customers churn dramatically more (~45% churn rate vs ~3% for 2-year)
2. **Tenure** — New customers (0-12 months) are the highest churn risk
3. **Internet Service** — Fiber optic users churn more (higher cost, more competition)
4. **Monthly Charges** — Higher bills → more price-sensitive churn
5. **Add-on Services** — Online security + Tech support increase customer stickiness
6. **Payment Method** — Electronic check users show less commitment

### 💡 Business Recommendations

Based on churn drivers identified:

1. **Offer long-term contract incentives** to month-to-month customers in months 1–6
   - This is where churn is highest
   - Offering a discount for annual/2-year contracts could lock in customers early

2. **Bundle add-on services** (security, tech support, backup)
   - Customers with 3+ add-ons show much lower churn
   - Create attractive bundles in first 3 months

3. **Proactive outreach for high-risk customers**
   - Flag: High monthly charge + short tenure + no add-ons
   - Offer targeted support or promotional discounts

4. **Review fiber optic pricing**
   - Compare against competitors
   - Consider tiered pricing or loyalty discounts for fiber customers

5. **Payment method optimization**
   - Encourage automatic payments (bank/card) over electronic checks
   - Offer small incentives for automatic payment enrollment

### 🚀 Further Improvements

- More extensive hyperparameter tuning (Optuna/Bayesian optimization)
- Model ensembling / stacking of top 3 models
- Threshold optimization for recall/precision tradeoff
- Target encoding for high-cardinality features
- Incorporating original IBM dataset for additional training data
- Time-series cross-validation if temporal data is available

### 📈 Competition Results

- **Private Score:** 0.91428
- **Ranking Impact:** This notebook achieved 31 upvotes and a **Bronze Medal** 🥉 on Kaggle
- **Lessons:** XGBoost proved to be the winner on this dataset, beating Random Forest by ~0.023 AUC

---

**If you found this notebook helpful, please consider:**
- Upvoting, following, and leaving feedback
- Exploring the original IBM dataset for comparison
- Experimenting with LightGBM or ensemble methods for potential improvements